# Unified Model Handoff — Ridge / LASSO / RF / XGBoost
Same 15 features, target, split, validation and portfolio rules. **No test tuning.**

In [ ]:
from pathlib import Path
import sys
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src'/'models.py').exists()),None)
if ROOT is None:raise RuntimeError('Open project root in VS Code')
sys.path.insert(0,str(ROOT))
from src.models import run_model
from src.backtest import portfolio_backtest,performance
from src.pipeline import FEATURES
FREQUENCY='monthly'  # monthly, quarterly, semiannual
MODEL='ridge'        # ridge, lasso, rf, xgb

## Train candidates on 2010–2020, select on 2021–2022, refit, then evaluate 2023–2025

In [ ]:
test_predictions,tuning,final_model=run_model(FREQUENCY,MODEL)
display(tuning)
print('Test rows:',len(test_predictions))
display(test_predictions[['decision_date','ticker','prediction','target_forward_return','pred_rank']].head(10))

## Portfolio comparison, costs, and stock-level holding P&L

In [ ]:
period_returns,trade_log,stock_pnl=portfolio_backtest(test_predictions,FREQUENCY,cost_bps=20)
summary=performance(period_returns,FREQUENCY)
display(summary)
display(trade_log.tail(10))
display(stock_pnl.tail(10))

## Charts: cumulative net wealth and validation loss

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
fig,ax=plt.subplots(figsize=(10,4))
for name,g in period_returns.groupby('strategy'):
    g=g.sort_values('decision_date')
    ax.plot(g.decision_date,(1+g.net_return).cumprod(),label=name)
ax.set_title('OOS cumulative wealth, net of assumed costs')
ax.legend();fig.tight_layout()
out=ROOT/'outputs'/'models'/FREQUENCY/MODEL
fig.savefig(out/'cumulative_wealth.png',dpi=160);plt.show()
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(range(len(tuning)),tuning.train_mse,label='Train MSE')
ax.plot(range(len(tuning)),tuning.validation_mse,label='Validation MSE')
ax.set_title('Training vs validation MSE by predeclared candidate')
ax.legend();fig.tight_layout();fig.savefig(out/'train_validation_mse.png',dpi=160);plt.show()

## Save reports

In [ ]:
period_returns.to_csv(out/'portfolio_returns.csv',index=False)
trade_log.to_csv(out/'rebalance_trade_log.csv',index=False)
stock_pnl.to_csv(out/'monthly_stock_pnl.csv',index=False)
summary.to_csv(out/'performance_summary.csv',index=False)
print('Saved to:',out)